In [1]:
import threading, time, random

data_pool = []
results =[]
lock = threading.Lock()
print_lock = threading.Lock()  # Für synchrone Ausgaben

In [2]:
# Zelle 2: Funktionen mit Fehlerbehandlung
def data_producer(worker_id):
    try:
        for i in range(3):
            with lock:
                new_data = f"Daten-{worker_id}-{i}"
                data_pool.append(new_data)
            
            with print_lock:
                print(f"🟢 PRODUCER {worker_id} erzeugt: {new_data}")
            
            time.sleep(random.uniform(0.1, 0.5))
    except Exception as e:
        with print_lock:
            print(f"❌ FEHLER in PRODUCER {worker_id}: {e}", file=sys.stderr)
        raise

In [3]:
def data_consumer(worker_id):
    try:
        processed = 0
        while processed < 3:
            time.sleep(random.uniform(0.2, 0.7))
            
            with lock:
                if data_pool:
                    data = data_pool.pop(0)
                    processed_data = data.upper()
                    results.append(processed_data)
                    processed += 1
                else:
                    continue
            
            with print_lock:
                print(f"🔴 CONSUMER {worker_id} verarbeitet: {data} -> {processed_data}")
    except Exception as e:
        with print_lock:
            print(f"❌ FEHLER in CONSUMER {worker_id}: {e}", file=sys.stderr)
        raise

In [22]:
def main():
    start_time = time.time()

    # threads erstellen
    producers = [threading.Thread(target = data_producer, args=(i,))  for i in range(2)]
    consumers = [threading.Thread(target = data_consumer, args=(i,))  for i in range(2)]
    
        # Threads starten
    for t in producers + consumers:
        t.start()
    
    # Auf Abschluss warten
    for t in producers + consumers:
        t.join()
    
    # Ergebnisse ausgeben
    print("\n" + "-"*50)
    print(f"Alle Threads abgeschlossen! Laufzeit: {time.time()-start_time:.2f}s")
    print(f"Endergebnisse: {results}")

In [ ]:
print("test")